# 📝 토픽 모델링 심화 과제 LV3(통합) — BERTopic 토픽 리포트·주제 탐색

> 이 단원에서 배운 **BERTopic 토픽 추출**과 **토픽 프로파일링·검증**을 각각 하나의 작은 **프로그램**으로 완성하는 통합 과제입니다. 문제마다 여러 `### N단계` 셀로 나뉘어 있고, **각 단계 셀에 그 단계에서 할 일(요구 변수·기대 형태·주의)이 자립적으로** 적혀 있어요.

## 풀이 방법
1. 문제마다 **1단계에서 데이터를 불러와** 같은 변수(`docs`·`emb`·`umap2d` 등)를 뒷단계로 이어 씁니다.
2. 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채우고, 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
3. **그래프 단계**는 자가채점이 없습니다 — 위 **완성 그래프(정답)** 와 같은 모양으로 그리세요.
4. **인사이트 서술 단계**는 서술형입니다 — 정답 노트북의 모범 서술과 비교하세요.

- 데이터는 문제 1 은 `data/reviews_sun.csv`(선크림 리뷰 545건), 문제 2 는 `data/reviews_mixed.csv`(자세밴드+선크림 리뷰 1445건)를 씁니다. **임베딩은 미리 계산돼** 있어 `np.load(...)` 로 불러옵니다(오프라인·재현).
- 한국어 키워드는 `korean_tokenizer` 가 책임집니다. 한글이 지워져 토픽을 못 뽑습니다.

화이팅!

아래 두 셀을 먼저 실행해 이 단원에 필요한 라이브러리·한국어 토크나이저·임베딩 모델을 준비하세요. (실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 토크나이저를 준비합니다.
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from kiwipiepy import Kiwi

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

# 지난 단원에서 배운 형태소 분석 — c-TF-IDF 키워드를 한국어 명사로 뽑는 토크나이저(자바가 필요없는 kiwipiepy)
kiwi = Kiwi()

# 불용어 — 11일차에서 배운 방식 그대로: 공개 일반 목록 + 이 데이터의 도메인 불용어
with open('data/stopwords_ko.json', encoding='utf-8') as f:
    STOPWORDS_GENERAL = set(json.load(f))     # 11일차에서 받아 둔 공개 목록 679개

# 이 데이터에서만 무의미한 고빈도어 — 빈도표를 보고 사람이 고른다(11일차 4절)
STOPWORDS_DOMAIN = {'제품', '구매', '사용', '정말', '진짜', '완전', '그냥', '너무', '정도', '많이'}

# 반대로 일반 목록이 '여기서는 의미 있는 말'까지 지우기도 한다 — 되살릴 단어
# ('아이'·'시간' 은 일반 불용어지만, 이 리뷰에서는 '아이에게 사 준 밴드'처럼 주제를 가른다)
KEEP_WORDS = {'아이', '시간'}
KOREAN_STOPWORDS = (STOPWORDS_GENERAL | STOPWORDS_DOMAIN) - KEEP_WORDS

def korean_tokenizer(text):
    """문서에서 의미있는 명사(2글자 이상)만 골라 돌려줍니다."""
    return [t.form for t in kiwi.tokenize(str(text))
            if t.tag.startswith('NN') and len(t.form) > 1 and t.form not in KOREAN_STOPWORDS]

In [ ]:
# [제공 코드] 지난 단원에서 배운 한국어 임베딩 모델을 불러옵니다(문장→768차원 벡터).
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# 임베딩·2D 좌표 — 저장된 파일이 있으면 그대로 쓰고, 없으면 지금 만들어 저장합니다
emb_path = 'data/reviews_sun_embeddings.npy'
umap_path = 'data/reviews_sun_umap2d.npy'
if not os.path.exists(emb_path):
    print('저장된 임베딩이 없어 지금 만듭니다 — 수 분 걸릴 수 있어요')
    texts = pd.read_csv('data/reviews_sun.csv')['text'].astype(str).tolist()
    np.save(emb_path, emb_model.encode(texts, show_progress_bar=False))
if not os.path.exists(umap_path):
    print('저장된 2차원 좌표가 없어 지금 만듭니다')
    np.save(umap_path, UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                            metric='cosine', random_state=42).fit_transform(np.load(emb_path)))

emb_path = 'data/reviews_mixed_embeddings.npy'
umap_path = 'data/reviews_mixed_umap2d.npy'
if not os.path.exists(emb_path):
    print('저장된 임베딩이 없어 지금 만듭니다 — 수 분 걸릴 수 있어요')
    texts = pd.read_csv('data/reviews_mixed.csv')['text'].astype(str).tolist()
    np.save(emb_path, emb_model.encode(texts, show_progress_bar=False))
if not os.path.exists(umap_path):
    print('저장된 2차원 좌표가 없어 지금 만듭니다')
    np.save(umap_path, UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                            metric='cosine', random_state=42).fit_transform(np.load(emb_path)))

## 1. 리뷰 토픽 리포트 생성기
**배경**: 한 선크림 상품에 달린 소비자 리뷰 545건에서 **사람들이 주로 말하는 주제**를 자동으로 뽑아, 토픽마다 `문서 수·대표 키워드·대표 리뷰` 를 담은 **리포트 표**를 만들고 CSV 로 저장하는 미니 프로그램을 완성합니다.

아래 각 `### N단계` 셀의 지시대로 **데이터 로드 → BERTopic 학습 → 리포트 표 만들기 → 저장** 순서로 이어서 풉니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | `docs` 545개, `emb` shape `(545, 768)` |
| 2단계 | 표준 BERTopic 학습 — `topics` 길이 545, 토픽 수(노이즈 제외) 3~15개 |
| 3단계 | `reduce_topics` 로 토픽 정리 — 정리 후 토픽 수 2~6개(≤ 정리 전) |
| 4단계 | 리포트 `report` 컬럼 `{topic_id, size, keywords, sample_doc}`, 행수 = 정리 후 토픽 수 |
| 5단계 | `size` 내림차순 정렬 후 `output/topic_report.csv` 저장 |
| 인사이트 | 뽑힌 토픽으로 리뷰어의 관심사 2~3문장 서술 |

### 1단계 — 데이터·임베딩 로드와 살펴보기
`data/reviews_sun.csv` 를 `df` 로 불러오고, 리뷰 본문 열(`text`)을 리스트로 만들어 `docs` 에, 미리 계산된 임베딩(`reviews_sun_embeddings.npy`)을 `emb` 에 담으세요. 그리고 리뷰 수·임베딩 shape 를 출력하고 `df.head()` 로 앞부분을 살펴보세요.

- **요구 변수**: `df`(원본), `docs`(리뷰 문자열 리스트, 545개), `emb`(넘파이 배열 `(545, 768)`).
- **주의**: 임베딩은 `np.load("data/reviews_sun_embeddings.npy")` 로 불러옵니다(직접 만들지 않음).

<details><summary>힌트</summary>

```text
접근방법:
- CSV 를 데이터프레임으로 읽고, 본문 열을 리스트로 바꿔 docs 에 담는다. 임베딩은 파일에서 불러온다.

세부구현:
1. read_csv 로 df 를 만든다
2. df 의 text 열을 리스트로 바꿔 docs 에 담는다
3. np.load 로 임베딩을 emb 에 담는다
4. len(docs) 와 emb.shape 를 출력하고 df.head() 로 살펴본다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(docs) == 545
assert emb.shape == (545, 768)
print("✅ 1단계 통과!")

### 2단계 — 표준 BERTopic 학습
이 단원에서 배운 **표준 BERTopic 구성**을 그대로 만들어 리뷰에서 토픽을 추출하세요. 미리 만든 `emb` 를 넘겨 학습을 빠르게 합니다.

> LV1 문제 7 에서는 `min_cluster_size=15` 로 잘게 뽑았습니다. 여기서는 **리포트로 읽을 굵은 주제**가 필요하니 **20** 으로 키웁니다 — 같은 데이터라도 이 값 하나로 토픽 구성이 달라집니다.

1. `umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)`
2. `hdbscan_model = HDBSCAN(min_cluster_size=20, metric="euclidean", prediction_data=True)`
3. `vectorizer_model = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)`
4. `topic_model = BERTopic(embedding_model=emb_model, umap_model=umap_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, verbose=False)`
5. `topics, probs = topic_model.fit_transform(docs, embeddings=emb)`

- **요구 변수**: `topic_model`(학습된 모델), `topics`(문서별 토픽 번호 리스트, 길이 545).
- **주의**: 노이즈(어느 토픽에도 안 들어간 리뷰)는 토픽 번호가 `-1` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- UMAP·HDBSCAN·CountVectorizer 세 구성요소를 만들어 BERTopic 에 끼우고, 미리 만든 임베딩으로 학습한다.

세부구현:
1. 지시대로 umap_model·hdbscan_model·vectorizer_model 을 만든다
2. 세 모델과 emb_model 을 BERTopic 에 넣는다
3. fit_transform 에 docs 와 embeddings=emb 를 넘겨 topics, probs 를 받는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(topics) == len(docs)
n_topics = int((topic_model.get_topic_info()['Topic'] != -1).sum())
assert 3 <= n_topics <= 15
print("✅ 2단계 통과!")

### 3단계 — 토픽 정리 (reduce_topics)
이 단원에서 배운 **토픽 수 조절**을 훈련합니다. BERTopic 이 뽑은 토픽이 7개 안팎으로 다소 많으니, `reduce_topics` 로 서로 비슷한 토픽을 합쳐 **5개 안팎**으로 정리합니다(리포트가 한눈에 들어오게).

1. 정리 **전** 토픽 수(노이즈 제외)를 `n_topics_before` 에 담습니다: `n_topics_before = int((topic_model.get_topic_info()['Topic'] != -1).sum())`.
2. `topic_model.reduce_topics(docs, nr_topics=5)` 로 토픽을 줄입니다. 정리 결과가 모델에 반영되므로, **문서별 토픽 번호도 바뀝니다** — `topics = topic_model.topics_` 로 갱신하세요.
3. 정리 **후** 토픽 수(노이즈 제외)를 `n_topics_after` 에 담습니다: `n_topics_after = int((topic_model.get_topic_info()['Topic'] != -1).sum())`.

- **요구 변수**: `n_topics_before`, `n_topics_after`, 그리고 갱신된 `topics`.
- **요구사항(느슨)**: 정리 후 토픽 수는 정리 전보다 **적거나 같고**, **2~6개** 범위입니다.
- **주의**: `nr_topics` 는 **노이즈(`-1`)를 포함한 수**입니다. `nr_topics=5` 로 주면 노이즈 1개 + 실제 토픽 4개 안팎이 됩니다(그래서 노이즈 제외 토픽 수는 5보다 작을 수 있어요). 정확한 토픽 수는 데이터·환경에 따라 조금 달라질 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 줄이기 전 토픽 수를 먼저 기록하고, reduce_topics 로 토픽을 합친 뒤 바뀐 토픽 번호를 다시 받아 온다.

세부구현:
1. get_topic_info() 의 Topic 열에서 -1 이 아닌 행 수를 세어 n_topics_before 를 구한다
2. reduce_topics 에 docs 와 nr_topics=5 를 넘긴다
3. topic_model.topics_ 로 갱신된 문서별 토픽 번호를 topics 에 다시 담는다
4. 같은 방식으로 정리 후 토픽 수를 세어 n_topics_after 를 구한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_topics_after <= n_topics_before
assert 2 <= n_topics_after <= 6
print("✅ 3단계 통과!")

### 4단계 — 토픽별 리포트 표 만들기
**3단계에서 정리한 모델**을 기준으로, 노이즈(`-1`)를 **제외한** 각 토픽마다 아래 네 값을 모아 DataFrame `report` 를 만드세요.

| 컬럼 | 내용 | 얻는 법 |
| --- | --- | --- |
| `topic_id` | 토픽 번호 | `get_topic_info()` 의 `Topic` |
| `size` | 그 토픽의 문서 수 | `get_topic_info()` 의 `Count` |
| `keywords` | 대표 키워드 **상위 6개**(단어 리스트) | `topic_model.get_topic(토픽번호)` 의 `(단어, 점수)` 중 단어 6개 |
| `sample_doc` | 대표 리뷰 **1개** | `topic_model.get_representative_docs(토픽번호)` 의 첫 번째 |

- **요구 변수**: `report`(위 4개 컬럼을 가진 DataFrame). 컬럼명은 표와 **똑같이** 쓰세요.
- **주의**: `get_topic_info()` 결과에는 노이즈 `-1` 행이 맨 위에 있습니다 — **`-1` 은 건너뜁니다.** `get_topic(...)` 은 `(단어, 점수)` 튜플 리스트라 단어만 골라 앞 6개를 씁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 토픽 정보를 훑으며 노이즈(-1)는 건너뛰고, 토픽마다 번호·문서수·키워드6·대표문서1을 모아 표로 만든다.

세부구현:
1. get_topic_info() 를 불러 Topic 열을 하나씩 본다
2. topic_id 가 -1 이면 건너뛴다
3. Count 로 size 를, get_topic(topic_id) 의 앞 6개 단어로 keywords 를 만든다
4. get_representative_docs(topic_id) 의 첫 문서를 sample_doc 로 담는다
5. 각 토픽을 딕셔너리로 모아 DataFrame report 를 만든다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
n_topics = int((topic_model.get_topic_info()['Topic'] != -1).sum())
assert set(report.columns) == {"topic_id", "size", "keywords", "sample_doc"}
assert len(report) == n_topics
assert int(report['size'].sum()) == int((pd.Series(topics) != -1).sum())
print("✅ 4단계 통과!")

### 5단계 — 문서 수 내림차순 정렬 후 CSV 저장
`report` 를 **문서 수(`size`) 내림차순**으로 정렬하고 `output/topic_report.csv` 로 저장한 뒤 화면에도 출력하세요.

1. `report = report.sort_values("size", ascending=False).reset_index(drop=True)`
2. 저장 폴더가 없을 수 있으니 `os.makedirs("output", exist_ok=True)` 로 먼저 만듭니다.
3. `report.to_csv("output/topic_report.csv", index=False)` 로 저장하고 `display(report)`.

- **요구사항**: `output/topic_report.csv` 파일이 생성되고, `report` 가 `size` 기준 내림차순으로 정렬돼야 합니다.
- **주의**: `os` 는 아직 import 하지 않았으니 이 셀에서 `import os` 를 먼저 하세요.

<details><summary>힌트</summary>

```text
접근방법:
- size 기준으로 내림차순 정렬하고, 저장 폴더를 만든 뒤 CSV 로 내보낸다.

세부구현:
1. sort_values 로 size 를 내림차순 정렬하고 인덱스를 새로 매긴다
2. os.makedirs 로 저장 폴더를 만든다(exist_ok=True)
3. to_csv 로 저장하고 display 로 표를 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import os
assert report["size"].is_monotonic_decreasing
assert os.path.exists("output/topic_report.csv")
print("✅ 5단계 통과!")

### 6단계 — 리포트에 **토픽 이름** 붙이기 (LLM)
지금 리포트의 `keywords` 는 `['크림', '피부', '화장', ...]` 처럼 **단어 나열**이라, 읽는 사람이 매번 해석해야 합니다. 교안 맛보기에서 본 **`representation_model`** 을 끼워 토픽마다 이름을 짓게 하고, 그 이름을 리포트의 새 컬럼 `topic_name` 으로 붙여 **사람이 바로 읽는 리포트**로 완성하세요.

1. 3단계에서 정리한 모델과 **같은 구성**(`min_cluster_size=20`)에 `representation_model` 만 더한 모델을 만들어 `named_model` 에 담고 학습한 뒤, `reduce_topics(docs, nr_topics=5)` 까지 똑같이 적용하세요(리포트와 토픽 수를 맞추기 위해서입니다).
```
from openai import OpenAI
from bertopic.representation import OpenAI as OpenAIRepresentation

prompt = ('다음은 한국어 선크림 리뷰 토픽입니다.\n'
          '대표 리뷰:\n[DOCUMENTS]\n'
          '키워드: [KEYWORDS]\n'
          '이 토픽의 이름을 한국어 10자 이내로 하나만 답하세요. 설명 없이 이름만.')
rep_model = OpenAIRepresentation(client=OpenAI(), model='gpt-4o-mini',
                                 chat=True, nr_docs=4, prompt=prompt)
```
2. `named_model.get_topic_info()` 의 `Topic`→`Name` 대응을 이용해, 5단계의 `report` 에 `topic_name` 컬럼을 추가한 표를 `report_named` 에 담으세요.
3. `output/topic_report_named.csv` 로 저장하고 `display` 로 확인하세요.

- **요구 변수**: `named_model`, `report_named`(기존 4컬럼 + `topic_name`).
> **키가 없으면 건너뜁니다.** `.env` 에 `OPENAI_API_KEY` 가 있을 때만 실행되고, 없으면 이 단계는 **채점에서 제외**됩니다(과금 없음).

<details><summary>힌트</summary>

```text
접근방법:
- 3단계와 같은 구성에 representation_model 만 더해 다시 학습하고, 같은 nr_topics 로 줄인다.
- 토픽 번호를 키로 이름을 map 해서 리포트에 컬럼을 붙인다.

세부구현:
1. .env 를 읽어 키가 없으면 건너뛴다
2. OpenAIRepresentation 을 만들어 표준 구성에 끼운 named_model 을 학습한다
3. reduce_topics 로 3단계와 같은 토픽 수로 맞춘다
4. get_topic_info 에서 Topic->Name 딕셔너리를 만든다
5. report 를 복사해 topic_id 를 그 딕셔너리로 map 한 topic_name 컬럼을 더한다
6. to_csv 로 저장하고 display 한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import os
if not os.getenv('OPENAI_API_KEY'):
    print('⏭️ 키가 없어 6단계는 채점에서 제외합니다.')
else:
    assert report_named is not None, 'report_named 가 없습니다'
    assert 'topic_name' in report_named.columns, 'topic_name 컬럼이 없습니다'
    assert len(report_named) == len(report), '리포트 행 수가 달라졌습니다'
    assert report_named['topic_name'].notna().all(), '이름이 비어 있는 토픽이 있습니다'
    assert os.path.exists('output/topic_report_named.csv')
    print('✅ 6단계 통과! 이름:', report_named['topic_name'].tolist())

### 인사이트 — 리뷰어의 관심사 읽기 (서술)
완성한 `report` 를 보고, 이 선크림 리뷰어들이 **주로 어떤 점에 관심을 두는지** 를 **2~3문장**으로 서술하세요. 문서 수가 많은 상위 토픽의 `keywords` 와 `sample_doc` 을 근거로 삼으세요.

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 리뷰어의 관심사를 2~3문장으로 서술하세요)*

## 2. 문서 주제 탐색기
**배경**: 두 상품군(자세밴드·선크림)의 리뷰가 **섞여 있는** 1445건에서, 라벨을 전혀 쓰지 않고 BERTopic 으로 **어떤 주제들이 들어 있는지 탐색**하는 프로그램입니다. 토픽마다 `문서 수·대표 키워드·대표 문서` 를 모은 **프로파일 표**를 만들고, 마지막에 **토픽 × 상품군 교차표**로 각 토픽이 실제로 무엇에 대한 이야기였는지 **눈으로 검증**합니다.

그림은 재현을 위해 미리 계산된 **2D 좌표**(`reviews_mixed_umap2d.npy`) 위에 그립니다(모든 컴퓨터에서 같은 지도).

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | `df` 1445행, `docs` 1445개, `emb` `(1445, 768)`, `umap2d` `(1445, 2)` |
| 2단계 | BERTopic 학습 — `topics` 길이 1445, 토픽 수(노이즈 제외) 3~10개, 노이즈 30% 이하 |
| 3단계 | 토픽별 대표 키워드 6개(`topic_keywords`)·대표 문서 1개(`topic_docs`) — 노이즈 제외 |
| 4단계 | 프로파일 `profile` 컬럼 `{topic_id, size, keywords, sample_doc}` + 노이즈는 `noise_size` 로 분리, `size` 합 + `noise_size` = 1445 |
| 5단계 | 토픽 산점도 + 교차표 `cross` · 30건 이상 토픽의 우세 비율 ≥ 0.7 · 검증 컬럼을 붙인 `profile_check` 저장 — 우세 상품군 비율 ≥ 0.7, 두 상품군이 각각 어떤 토픽의 우세군 |
| 인사이트 | 교차표와 대표 키워드를 근거로 토픽의 정체를 서술 |

### 1단계 — 데이터·임베딩·좌표 로드와 살펴보기
`data/reviews_mixed.csv` 를 `df` 로, 리뷰 본문 열(`text`)을 리스트로 만들어 `docs` 에, 임베딩(`reviews_mixed_embeddings.npy`)을 `emb` 로, 재현용 2D 좌표(`reviews_mixed_umap2d.npy`)를 `umap2d` 로 불러오세요. 그리고 리뷰 수·shape 와 **상품군 분포**(`product_type` 값별 개수)를 살펴보세요.

- **요구 변수**: `df`(1445행), `docs`(1445개), `emb`(`(1445, 768)`), `umap2d`(`(1445, 2)`).
- **주의**: `product_type` 은 **마지막에 눈으로 검증할 때만** 쓰는 라벨입니다 — 토픽 탐색(2~4단계)에는 쓰지 않고 리뷰 본문과 임베딩만 씁니다.

<details><summary>힌트</summary>

```text
접근방법:
- CSV·임베딩·2D 좌표를 각각 불러오고, 본문 열을 리스트로 만든 뒤 크기와 상품군 분포를 확인한다.

세부구현:
1. read_csv 로 df 를 만들고 text 열을 리스트로 바꿔 docs 에 담는다
2. np.load 로 emb 와 umap2d 를 불러온다
3. len(df)·emb.shape·umap2d.shape 를 출력하고 product_type 의 value_counts 를 본다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(df) == 1445
assert len(docs) == 1445
assert emb.shape == (1445, 768)
assert umap2d.shape == (1445, 2)
print("✅ 1단계 통과!")

### 2단계 — BERTopic 으로 주제 탐색하기
라벨을 쓰지 않고 리뷰 1445건에서 **어떤 주제들이 들어 있는지** BERTopic 으로 찾습니다. 문제 1 과 같은 표준 구성이되, 작은 주제까지 잡아내도록 **`min_cluster_size=10`** 으로 조금 잘게 봅니다.

1. `umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42)`
2. `hdbscan_model = HDBSCAN(min_cluster_size=10, metric="euclidean", prediction_data=True)`
3. `vectorizer_model = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)`
4. `topic_model = BERTopic(embedding_model=emb_model, umap_model=umap_model, hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model, verbose=False)`
5. `topics, probs = topic_model.fit_transform(docs, embeddings=emb)`

그리고 **토픽 수(노이즈 제외)** 를 `n_topics` 에, **노이즈(`-1`) 문서 수**를 `noise_size` 에 담아 출력하세요.

- **요구 변수**: `topic_model`, `topics`(길이 1445), `n_topics`, `noise_size`.
- **요구사항**: 토픽 수는 **3~10개**, 노이즈는 전체의 **30% 이하**로 나옵니다(실측 기준값: 토픽 6개, 노이즈 65건 ≈ 4.5%).
- **주의**: 노이즈 행 `-1` 이 있을 수도, 없을 수도 있으므로 토픽 수는 `(get_topic_info()["Topic"] != -1).sum()` 처럼 **`-1` 을 직접 걸러** 세는 것이 안전합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 과 같은 구성으로 BERTopic 을 만들되 min_cluster_size 만 10 으로 주고, 미리 만든 임베딩으로 학습한다.

세부구현:
1. 지시대로 umap_model·hdbscan_model·vectorizer_model 을 만든다
2. 세 모델과 emb_model 을 BERTopic 에 넣는다
3. fit_transform 에 docs 와 embeddings=emb 를 넘겨 topics, probs 를 받는다
4. get_topic_info 의 Topic 열에서 -1 이 아닌 행 수를 n_topics 에 담는다
5. topics 중 -1 의 개수를 세어 noise_size 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(topics) == 1445
assert 3 <= n_topics <= 10
assert noise_size <= 1445 * 0.3
print("✅ 2단계 통과!")

### 3단계 — 토픽별 대표 키워드·대표 문서 뽑기
찾은 토픽마다 **대표 키워드 6개**와 **대표 문서 1개**를 모아 두 개의 딕셔너리에 담으세요. 노이즈(`-1`)는 주제가 아니므로 **건너뜁니다**.

- `topic_keywords`: `{토픽번호: [단어 6개]}` — `topic_model.get_topic(번호)` 는 `(단어, 점수)` 튜플 리스트라 **단어만** 골라 앞 6개를 씁니다.
- `topic_docs`: `{토픽번호: 대표 문서 1개(문자열)}` — `topic_model.get_representative_docs(번호)` 의 첫 번째.

- **요구 변수**: `topic_keywords`, `topic_docs`(둘 다 키는 노이즈를 뺀 토픽 번호, 개수 = `n_topics`).
- **주의**: 키워드는 반드시 6개여야 하고, 대표 문서는 `docs` 안에 실제로 있는 리뷰여야 합니다. 토픽 번호는 `get_topic_info()["Topic"]` 에서 얻으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 토픽 번호를 하나씩 돌며 노이즈는 건너뛰고, 키워드 6개와 대표 문서 1개를 각각 딕셔너리에 담는다.

세부구현:
1. 빈 딕셔너리 두 개를 만든다
2. get_topic_info() 의 Topic 열을 하나씩 본다
3. -1 이면 건너뛴다
4. get_topic(번호) 의 앞 6개에서 단어만 뽑아 topic_keywords 에 담는다
5. get_representative_docs(번호) 의 첫 문서를 topic_docs 에 담는다
6. 토픽마다 번호·키워드·대표 문서 앞부분을 출력해 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(topic_keywords) == set(topic_docs)
assert len(topic_keywords) == n_topics
assert -1 not in topic_keywords
assert all(len(v) == 6 for v in topic_keywords.values())
assert all(isinstance(d, str) and d in docs for d in topic_docs.values())
print("✅ 3단계 통과!")

### 4단계 — 토픽 프로파일 표 만들기 (노이즈는 따로)
3단계에서 모은 키워드·대표 문서에 **문서 수**를 더해, 노이즈를 **뺀** 토픽만으로 DataFrame `profile` 을 만들고 **`size` 내림차순**으로 정렬하세요. 노이즈는 표에 섞지 않고 **따로 크기와 비율만** 출력합니다.

| 컬럼 | 내용 | 얻는 법 |
| --- | --- | --- |
| `topic_id` | 토픽 번호 | `topic_keywords` 의 키 |
| `size` | 그 토픽의 문서 수 | `topics` 에서 그 번호의 개수 |
| `keywords` | 대표 키워드 6개 | 3단계 `topic_keywords` |
| `sample_doc` | 대표 문서 1개 | 3단계 `topic_docs` |

- **요구 변수**: `profile`(위 4개 컬럼, 행수 = `n_topics`, `size` 내림차순), 그리고 노이즈 비율 `noise_ratio`(= `noise_size` / 1445).
- **요구사항**: 노이즈를 뺐으므로 `profile["size"].sum() + noise_size == 1445` 가 됩니다.
- **주의**: 노이즈 `-1` 은 **주제가 아니라 '어디에도 안 든 문서 더미'** 라서 표에 넣으면 가장 큰 토픽처럼 보입니다. 크기 세기는 `pd.Series(topics).value_counts()` 가 편합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 토픽 번호별 문서 수를 세어 두고, 3단계 딕셔너리와 합쳐 표를 만든 뒤 크기순으로 정렬한다. 노이즈는 표 밖에서 따로 출력한다.

세부구현:
1. pd.Series(topics).value_counts() 로 토픽별 문서 수를 센다
2. topic_keywords 의 키를 돌며 번호·문서수·키워드·대표문서를 딕셔너리로 모은다
3. DataFrame 으로 만들고 size 내림차순으로 정렬한 뒤 인덱스를 새로 매긴다
4. noise_size / len(topics) 로 noise_ratio 를 구해 따로 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(profile.columns) == {"topic_id", "size", "keywords", "sample_doc"}
assert len(profile) == n_topics
assert -1 not in set(profile["topic_id"])
assert profile["size"].is_monotonic_decreasing
assert int(profile['size'].sum()) + noise_size == 1445
assert abs(noise_ratio - noise_size / 1445) < 1e-9
print("✅ 4단계 통과!")

### 5단계 — 토픽 산점도와 토픽 × 상품군 교차표
먼저 `umap2d` 좌표 위에 리뷰를 **토픽 색으로 칠한 산점도**를 그려 토픽이 지도 위에서 어떻게 나뉘는지 눈으로 확인하세요(노이즈는 **회색**으로 따로). 그다음 각 토픽이 실제로 어느 상품군(`product_type`)의 이야기였는지 **교차표**로 확인합니다.

1. 산점도: `plt.figure()` 로 새 그림을 연 뒤, `topics` 를 `np.array` 로 바꿔 **노이즈(`-1`)는 `color="lightgray"` 로 먼저**, 나머지는 `c=토픽번호, cmap="tab10"` 으로 그리고 제목·축 이름·범례를 답니다. **이 그래프는 자가채점이 없습니다** — 아래 완성 그래프처럼 그리세요.
2. 교차표: `cross = pd.crosstab(pd.Series(topics, name="topic"), df["product_type"])` 로 만들고 `display(cross)` 로 봅니다.
3. 노이즈를 뺀 교차표를 `cross_t` 에 담고(−1 행 제외), 토픽마다 **우세 상품군**을 `dominant` 에, 그 **비율**(우세 상품군 문서 수 ÷ 그 토픽 전체 문서 수)을 `share` 에 구해 출력하세요.
4. **검증 결과를 프로파일에 붙입니다.** 4단계의 `profile` 에 `dominant_type`(우세 상품군)·`dominant_share`(우세 비율) 두 컬럼을 더한 표를 `profile_check` 에 담고, `output/topic_profile_check.csv` 로 저장하세요. (`topic_id` 를 키로 `dominant`·`share` 값을 가져오면 됩니다.)

- **요구 변수**: `cross`(교차표), `cross_t`(노이즈 −1 행을 뺀 교차표), `dominant`(토픽별 우세 상품군), `share`(토픽별 우세 비율), `profile_check`(프로파일 + 검증 2컬럼). **다섯 개 모두** 자가채점이 확인합니다.

> **문제 1 의 리포트와 무엇이 다른가**: 문제 1 은 "무슨 주제가 있나"를 정리한 표였습니다. 여기서는 그 위에 **"그 주제가 실제로 어느 상품군의 이야기였나"** 를 붙여, 라벨 없이 찾은 토픽이 맞았는지까지 담은 **검증된 프로파일**을 산출물로 남깁니다.
- **요구사항**: **문서 30건 이상인 토픽**은 우세 비율이 **0.7 이상**이고, **두 상품군이 각각 최소 한 토픽의 우세 상품군**으로 나타납니다(실측: 30건 이상 토픽의 최소 비율 0.903, 전체 최소 0.762). 아주 작은 토픽은 문서 몇 개만 옮겨져도 비율이 크게 흔들려 채점에서 뺍니다.
- **주의**: 노이즈 행 `-1` 을 빼고 봐야 합니다. `-1` 은 여러 주제가 섞인 더미라 우세 상품군을 따지는 것이 의미가 없어요.

<details><summary>힌트</summary>

```text
접근방법:
- 노이즈를 회색으로 깔고 그 위에 토픽 색 산점도를 그린 뒤, 토픽과 상품군의 교차표를 만들어 토픽마다 우세 상품군과 비율을 구한다.

세부구현:
1. topics 를 np.array 로 바꿔 -1 인지 아닌지 마스크를 만든다
2. plt.figure 로 새 그림을 열고 노이즈 점을 lightgray 로 먼저 그린다
3. 나머지 점을 c=그 토픽 번호, cmap='tab10' 으로 그린다
4. 제목·축 이름·범례를 달고 보여 준다
5. pd.crosstab 으로 토픽 × product_type 교차표 cross 를 만들어 본다
6. cross 에서 -1 행을 뺀 뒤 idxmax(axis=1) 로 우세 상품군, max/sum 으로 비율을 구한다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q2_scatter.png" width="560"/>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert int(cross.values.sum()) == 1445
assert set(cross.columns) == {"선크림", "자세밴드"}
assert len(cross[cross.index != -1]) == n_topics
big = cross_t.sum(axis=1) >= 30      # 아주 작은 토픽은 비율이 흔들려 채점에서 제외
assert (share[big] >= 0.7).all()     # 큰 토픽은 한쪽 상품군이 뚜렷이 우세
assert set(dominant) == {"선크림", "자세밴드"}   # 두 상품군이 각각 어떤 토픽의 우세군
import os
# 검증 결과가 붙은 최종 프로파일
assert set(profile_check.columns) == {"topic_id", "size", "keywords", "sample_doc", "dominant_type", "dominant_share"}, 'profile_check 는 profile 4컬럼 + dominant_type·dominant_share 여야 합니다'
assert len(profile_check) == n_topics
assert set(profile_check["dominant_type"]) <= {"선크림", "자세밴드"}
assert (profile_check["dominant_share"] >= 0.5).all(), '우세 비율은 정의상 0.5 이상이어야 합니다'
assert os.path.exists("output/topic_profile_check.csv")
print("✅ 5단계 통과!")

### 인사이트 — 발견한 주제 읽기 (서술)
위 **교차표**와 `profile` 의 **대표 키워드·대표 문서**를 근거로, 라벨 없이 뽑은 토픽들이 무엇에 대한 이야기였는지 **2~3문장**으로 서술하세요. 큰 토픽이 상품군을 어떻게 갈랐는지, 작은 토픽은 어떤 주제였는지를 함께 짚으면 좋습니다.

> 이 단계는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 발견한 주제들이 무엇에 대한 이야기였는지 2~3문장으로 서술하세요)*